
# 🚦 Projeto de IA Aplicada à Mobilidade Urbana
## Grupo 6 — Ciência de Dados e Inteligência Artificial

### 📌 Tema
**Como melhorar o trânsito das cidades utilizando Inteligência Artificial**

### 👥 Integrantes
- Nome Integrante 1
- Nome Integrante 2
- Nome Integrante 3
- Nome Integrante 4

---

# 🎯 Objetivo do Projeto

Este projeto utiliza técnicas de Ciência de Dados e Machine Learning para:
- analisar padrões de tráfego;
- identificar horários críticos;
- detectar gargalos urbanos;
- construir modelos preditivos;
- gerar insights estratégicos para cidades inteligentes.

O notebook foi desenvolvido para execução completa no Google Colab.


In [ ]:

# ==========================================================
# INSTALAÇÃO DE BIBLIOTECAS
# ==========================================================

!pip -q install gdown xgboost plotly missingno folium

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import missingno as msno
import folium
import gdown
import os
import warnings

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb

warnings.filterwarnings('ignore')

sns.set_style('whitegrid')

print("✅ Bibliotecas carregadas com sucesso.")



# 📥 Download e Carregamento do Dataset

Nesta etapa o dataset é baixado automaticamente do Google Drive.

O sistema realiza:
- download automático;
- detecção de separador CSV;
- verificação de erros;
- exibição inicial dos dados.


In [ ]:

# ==========================================================
# DOWNLOAD E CARREGAMENTO DO DATASET
# ==========================================================

FILE_ID = "1aCQg7WYuiUEwvDOtsuuyyU_CB2SRxuVw"
OUTPUT = "dataset.csv"

url = f"https://drive.google.com/uc?id={FILE_ID}"

gdown.download(url, OUTPUT, quiet=False)

assert os.path.exists(OUTPUT), "❌ Dataset não foi baixado."

try:

    df = pd.read_csv(
        OUTPUT,
        sep=';',
        encoding='latin1',
        low_memory=False
    )

    if len(df.columns) == 1:

        df = pd.read_csv(
            OUTPUT,
            sep=',',
            encoding='latin1',
            low_memory=False
        )

    print("✅ Dataset carregado com sucesso.")
    print(f"Dimensões: {df.shape}")

    display(df.head())

except Exception as e:

    print("❌ Erro ao carregar dataset:")
    print(e)



# 🔍 Análise Inicial do Dataset

Esta etapa permite compreender:
- tipos de variáveis;
- dados ausentes;
- estrutura geral;
- qualidade dos dados.


In [ ]:

assert 'df' in locals(), "❌ DataFrame não carregado."

print(df.info())

print("\nValores ausentes:")
print(df.isnull().sum())

display(df.describe(include='all'))

msno.matrix(df)
plt.show()



# 🕒 Engenharia Temporal

Nesta etapa são extraídas informações temporais importantes:
- hora do dia;
- dia da semana;
- padrões temporais de mobilidade.


In [ ]:

possible_date_cols = [c for c in df.columns if 'data' in c.lower()]

if len(possible_date_cols) > 0:

    date_col = possible_date_cols[0]

    df[date_col] = pd.to_datetime(
        df[date_col],
        errors='coerce',
        dayfirst=True
    )

    df['hora'] = df[date_col].dt.hour
    df['dia_da_semana'] = df[date_col].dt.day_name()

    print(f"✅ Coluna temporal utilizada: {date_col}")

else:

    print("⚠ Nenhuma coluna de data encontrada.")



# 🔥 Heatmap de Correlação

O heatmap mostra relações entre variáveis numéricas.

## Interpretação
- cores fortes representam maior correlação;
- ajuda a identificar fatores relevantes;
- permite entender padrões urbanos.


In [ ]:

numeric_df = df.select_dtypes(include=['int64', 'float64'])

if numeric_df.shape[1] > 1:

    corr = numeric_df.corr()

    plt.figure(figsize=(14,10))

    sns.heatmap(
        corr,
        cmap='coolwarm',
        annot=True,
        fmt='.2f'
    )

    plt.title("Heatmap de Correlação")
    plt.show()

else:

    print("⚠ Poucas colunas numéricas disponíveis.")



# ⏰ Distribuição do Tráfego por Hora

Este gráfico identifica horários críticos de congestionamento.

## O que observar?
- picos de fluxo;
- horários críticos;
- comportamento urbano diário.


In [ ]:

if 'hora' in df.columns:

    plt.figure(figsize=(14,6))

    sns.histplot(
        df['hora'].dropna(),
        bins=24,
        kde=True
    )

    plt.title("Distribuição de Tráfego por Hora")
    plt.xlabel("Hora")
    plt.ylabel("Frequência")
    plt.show()

else:

    print("⚠ Coluna 'hora' não encontrada.")



# 🚗 Distribuição Estatística da Velocidade

O boxplot identifica:
- mediana;
- dispersão;
- anomalias;
- congestionamentos severos.


In [ ]:

velocidade_col = None

for c in df.columns:
    if 'veloc' in c.lower():
        velocidade_col = c
        break

if velocidade_col:

    df[velocidade_col] = pd.to_numeric(
        df[velocidade_col],
        errors='coerce'
    )

    plt.figure(figsize=(12,6))

    sns.boxplot(x=df[velocidade_col])

    plt.title("Distribuição da Velocidade")
    plt.xlabel("Velocidade")
    plt.show()

else:

    print("⚠ Coluna de velocidade não encontrada.")



# 📍 Dashboard Geográfico

O mapa mostra regiões urbanas com maior fluxo de tráfego.

## Aplicação
- planejamento urbano;
- análise espacial;
- mobilidade inteligente.


In [ ]:

lat_col = None
lon_col = None

for c in df.columns:

    if 'lat' in c.lower():
        lat_col = c

    if 'lon' in c.lower():
        lon_col = c

if lat_col and lon_col:

    sample_df = df.sample(min(3000, len(df)))

    fig = px.scatter_mapbox(
        sample_df,
        lat=lat_col,
        lon=lon_col,
        zoom=10,
        title="Mapa Geográfico do Fluxo Urbano"
    )

    fig.update_layout(mapbox_style="open-street-map")

    fig.show()

else:

    print("⚠ Coordenadas geográficas não encontradas.")



# 🤖 Preparação para Machine Learning

Nesta etapa:
- os dados são preparados;
- as variáveis são organizadas;
- o conjunto de treino é criado.


In [ ]:

assert 'df' in locals(), "❌ DataFrame indisponível."

velocidade_col = None
lat_col = None
lon_col = None

for c in df.columns:

    if 'veloc' in c.lower():
        velocidade_col = c

    if 'lat' in c.lower():
        lat_col = c

    if 'lon' in c.lower():
        lon_col = c

assert velocidade_col is not None, "❌ Coluna velocidade não encontrada."

assert lat_col is not None, "❌ Latitude não encontrada."

assert lon_col is not None, "❌ Longitude não encontrada."

df_ml = df.copy()

df_ml[velocidade_col] = pd.to_numeric(
    df_ml[velocidade_col],
    errors='coerce'
)

df_ml[lat_col] = pd.to_numeric(
    df_ml[lat_col],
    errors='coerce'
)

df_ml[lon_col] = pd.to_numeric(
    df_ml[lon_col],
    errors='coerce'
)

df_ml = df_ml.dropna(
    subset=[velocidade_col, lat_col, lon_col]
)

features = [lat_col, lon_col]

if 'hora' in df_ml.columns:
    features.append('hora')

X = df_ml[features]
y = df_ml[velocidade_col]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("✅ Dados de treino preparados.")
print(X_train.shape)



# 🧠 Treinamento do Modelo de IA

Modelo utilizado:
- XGBoost Regressor

Objetivo:
Prever comportamento do tráfego urbano.


In [ ]:

model = xgb.XGBRegressor(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    random_state=42
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"R2 Score: {r2:.4f}")
print(f"RMSE: {rmse:.4f}")



# 📈 Comparação Real vs Predito

Este gráfico compara:
- valores reais;
- valores previstos pelo modelo.

## Interpretação
Quanto mais próximos os pontos estiverem da diagonal:
- melhor o desempenho do modelo.


In [ ]:

plt.figure(figsize=(10,6))

plt.scatter(
    y_test,
    y_pred,
    alpha=0.5
)

plt.xlabel("Valores Reais")
plt.ylabel("Valores Preditos")
plt.title("Real vs Predito")

plt.show()



# 📊 Importância das Variáveis

O gráfico abaixo mostra:
- quais fatores tiveram maior impacto nas previsões.

## Benefício
Permite explicar o comportamento da IA.


In [ ]:

importance = pd.DataFrame({
    'Variavel': X_train.columns,
    'Importancia': model.feature_importances_
})

importance = importance.sort_values(
    by='Importancia',
    ascending=False
)

plt.figure(figsize=(12,6))

sns.barplot(
    data=importance,
    x='Importancia',
    y='Variavel'
)

plt.title("Importância das Variáveis")
plt.show()



# ✅ Conclusões Estratégicas

Com base nas análises realizadas foi possível:

- identificar horários críticos;
- compreender padrões urbanos;
- detectar possíveis gargalos;
- aplicar Machine Learning em mobilidade urbana;
- gerar insights para cidades inteligentes.

## Aplicações futuras
- semáforos inteligentes;
- previsão de congestionamentos;
- roteamento dinâmico;
- gestão pública baseada em dados.
